## Raw file analisis

In [2]:
!pwd

/home/baptvit/Documents/mestrado/master-experiments/analisis


## Count of the tokens, caracteres and fhir resources

In [3]:
path_file = """/home/baptvit/Documents/mestrado/master-experiments/master_experiments/fhir_data/stanford_llm_on_fhir/Allen322_Ferry570_ad134528-56a5-35fd-c37f-466ff119c625.json"""

In [4]:
import json
import os
import tiktoken

def count_tokens_and_characters(json_file_path, model_name="gpt-3.5-turbo"):
    """
    Counts the number of tokens and characters in a JSON file.

    Args:
        json_file_path (str): The path to the JSON file.
        model_name (str): The model name for tokenization. Default is "gpt-3.5-turbo".

    Returns:
        dict: A dictionary containing the number of tokens and characters.
    """
    if not os.path.exists(json_file_path):
        raise FileNotFoundError(f"File not found: {json_file_path}")

    try:
        with open(json_file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON file: {e}")

    # Serialize the JSON data back to a string for token and character counting
    json_string = json.dumps(data, ensure_ascii=False)

    # Count characters
    character_count = len(json_string)

    # Initialize tiktoken encoder
    try:
        encoder = tiktoken.encoding_for_model(model_name)
    except KeyError:
        raise ValueError(f"Model '{model_name}' not supported by tiktoken.")

    # Tokenize the JSON string
    tokens = encoder.encode(json_string)
    token_count = len(tokens)

    return {
        "tokens": token_count,
        "characters": character_count
    }

In [52]:
result = count_tokens_and_characters(path_file)

In [53]:
result

{'tokens': 908412, 'characters': 2358050}

In [5]:
def count_fhir_resources(file_path):
    """
    Counts the number of FHIR resources in a .json FHIR file.

    Parameters:
        file_path (str): The path to the FHIR .json file.

    Returns:
        int: The number of FHIR resources found in the file.
    """
    try:
        # Open and load the JSON file
        with open(file_path, 'r', encoding='utf-8') as file:
            fhir_data = json.load(file)

        # Check if it's a Bundle resource
        if fhir_data.get('resourceType') == 'Bundle':
            # Count the entries in the Bundle
            return len(fhir_data.get('entry', []))
        else:
            # For standalone FHIR resources, return 1
            return 1
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format: {e}")
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file_path}")
    except Exception as e:
        raise RuntimeError(f"An error occurred while processing the file: {e}")

In [55]:
count_fhir_resources(path_file)

1235

## Count in neo4j

In [14]:
from neo4j import GraphDatabase

def count_tokens_in_neo4j_nodes(consumer_id, uri, user, password, model_name="gpt-3.5-turbo"):
    """
    Connects to a Neo4j database, retrieves all nodes, extracts their `text` attributes,
    and counts the total number of tokens in those attributes.

    Parameters:
        uri (str): The URI of the Neo4j database (e.g., "bolt://localhost:7687").
        user (str): The username for Neo4j authentication.
        password (str): The password for Neo4j authentication.

    Returns:
        tuple: A tuple containing:
            - List of `text` attributes from all nodes.
            - Total number of tokens in the `text` attributes.
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))

    try:
        with driver.session() as session:
            # Query to retrieve all nodes with their `text` attributes
            query = f"MATCH (n) WHERE n.consumer_id = '{consumer_id}' RETURN n.text AS text"
            result = session.run(query)

            # Extract the `text` attributes
            texts = [record["text"] for record in result]

            encoder = tiktoken.encoding_for_model(model_name)

            # Tokenize and count tokens
            # tokens = encoder.encode(json_string)
            # token_count = len(tokens)
    
            total_tokens = sum(len(encoder.encode(text)) for text in texts if text)

            return total_tokens

    except Exception as e:
        raise RuntimeError(f"An error occurred: {e}")

    finally:
        driver.close()


In [15]:
# Example usage:
uri = "bolt://localhost:7687"
user = "neo4j"
password = "password"
consumer_id = 'Jacklyn830_Veum823_e0e1f21a-22a7-d166-7bb1-63f6bbce1a32'
total_tokens = count_tokens_in_neo4j_nodes(consumer_id, uri, user, password)


In [16]:
total_tokens

558554

### Nodes count

In [ ]:
MATCH (n)
WHERE n.consumer_id = 'Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d'
RETURN COUNT(n)

### Relationship count

In [ ]:
MATCH (n)-[r]-()
WHERE n.consumer_id = 'Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d'
RETURN COUNT(r) AS relationshipCount

In [1]:
import json
import os
import tiktoken

def count_tokens(json_string, model_name="gpt-4"):
    """
    Counts the number of tokens and characters in a JSON file.

    Args:
        json_file_path (str): The path to the JSON file.
        model_name (str): The model name for tokenization. Default is "gpt-3.5-turbo".

    Returns:
        dict: A dictionary containing the number of tokens and characters.
    """
    # Initialize tiktoken encoder
    try:
        encoder = tiktoken.encoding_for_model(model_name)
    except KeyError:
        raise ValueError(f"Model '{model_name}' not supported by tiktoken.")

    # Tokenize the JSON string
    tokens = encoder.encode(json_string)
    token_count = len(tokens)

    return token_count

In [2]:
count_tokens("hello_world")

2